In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test1') \
    .getOrCreate()

26/06/24 21:18:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [37]:
# This is the functionality that we want to achive using RDD.

'''SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2'''

"SELECT \n    date_trunc('hour', lpep_pickup_datetime) AS hour, \n    PULocationID AS zone,\n\n    SUM(total_amount) AS amount,\n    COUNT(1) AS number_records\nFROM\n    green\nWHERE\n    lpep_pickup_datetime >= '2020-01-01 00:00:00'\nGROUP BY\n    1, 2"

In [3]:
df_green = spark.read.parquet('data/pq/green/*/*')

26/06/24 21:20:07 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

In [4]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [5]:
from datetime import datetime

In [9]:
start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [12]:
rows = rdd.take(10)
rows

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 9, 40, 17), PULocationID=42, total_amount=7.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 2, 10, 15, 8), PULocationID=7, total_amount=9.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 6, 7, 28, 37), PULocationID=89, total_amount=18.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 2, 9, 50, 22), PULocationID=75, total_amount=12.06),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 7, 9, 37, 16), PULocationID=41, total_amount=7.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 5, 17, 36, 21), PULocationID=260, total_amount=12.05),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 6, 8, 25, 30), PULocationID=129, total_amount=23.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 3, 20, 56, 35), PULocationID=97, total_amount=11.76),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 6, 22, 53, 46), PULocationID=226, total_amount=13.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 

In [11]:
rdd.filter(filter_outliers).take(1)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 9, 40, 17), PULocationID=42, total_amount=7.3)]

In [13]:
rows = rdd.take(10)
row = rows[0]

In [23]:
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 9, 40, 17), PULocationID=42, total_amount=7.3)

In [24]:
def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [25]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

In [26]:
from collections import namedtuple

In [27]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [28]:
def unwrap(row):
    return RevenueRow(
        hour=row[0][0], 
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

In [29]:
from pyspark.sql import types

In [30]:
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [31]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema) 

In [33]:
df_result.write.parquet('tmp/green-revenue' ,mode="overwrite")

26/06/24 23:18:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [ ]:
# The above process happens when you simply call the query on da